# Model Development and Evaluation

This notebook focuses on training and evaluating machine learning models for credit default prediction.

The objective is to identify patterns associated with loan repayment behavior and compare multiple algorithms using appropriate classification metrics.

Model performance will primarily be assessed using ROC-AUC due to the imbalanced nature of the target variable.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [ ]:
train_df = pd.read_csv(
    "../data/processed/train_processed.csv"
)

train_df.shape

In [ ]:
train_df.head()

In [ ]:
X = train_df.drop(
    columns=["TARGET"]
)

y = train_df["TARGET"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

## Train-Test Split

The dataset is divided into training and testing subsets.

Stratified sampling is used to preserve the original class distribution and ensure reliable model evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

In [ ]:
print(
    y_train.value_counts(normalize=True) * 100
)

### Observations

The dataset exhibits significant class imbalance, with 91.93% non-default cases and 8.07% default cases.

To maintain this distribution during model development, stratified sampling was applied when creating the training and testing datasets.

Since a model could achieve high accuracy by simply predicting the majority class, evaluation will focus on ROC-AUC and other classification metrics that better measure the ability to distinguish between defaulters and non-defaulters.

This approach aligns with industry practices in credit risk modeling, where identifying high-risk applicants is more important than maximizing overall accuracy.

## Baseline Model: Logistic Regression

Logistic Regression is used as a baseline classification model.

Because Logistic Regression is sensitive to feature scale, numerical features are standardized before training.

The baseline model provides a reference point against which more advanced algorithms such as XGBoost and LightGBM can be compared.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
log_preds = log_model.predict(X_test_scaled)

log_probs = log_model.predict_proba(
    X_test_scaled
)[:, 1]

In [ ]:
log_auc = roc_auc_score(
    y_test,
    log_probs
)

print("Logistic Regression ROC-AUC:", log_auc)

In [ ]:
print(
    classification_report(
        y_test,
        log_preds
    )
)

In [ ]:
confusion_matrix(
    y_test,
    log_preds
)

## Addressing Class Imbalance

The baseline Logistic Regression model achieved a reasonable ROC-AUC score but demonstrated very poor recall for the default class.

To improve the model's ability to identify defaulters, class weighting is introduced. This assigns greater importance to minority-class observations during training and helps reduce bias toward the majority class.

In [ ]:
balanced_log_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

balanced_log_model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
balanced_preds = balanced_log_model.predict(
    X_test_scaled
)

balanced_probs = (
    balanced_log_model.predict_proba(
        X_test_scaled
    )[:, 1]
)

In [ ]:
balanced_auc = roc_auc_score(
    y_test,
    balanced_probs
)

print("Balanced Logistic Regression ROC-AUC:", balanced_auc)

In [ ]:
print(
    classification_report(
        y_test,
        balanced_preds
    )
)

In [ ]:
confusion_matrix(
    y_test,
    balanced_preds
)

## Logistic Regression Evaluation Summary

Two Logistic Regression models were evaluated.

The baseline model achieved a ROC-AUC score of approximately 0.75 but identified very few default cases due to the strong class imbalance present in the dataset.

Introducing class weighting substantially improved recall for the default class (from 1% to 68%), demonstrating the importance of imbalance-aware training strategies. While precision decreased, the model became significantly more effective at identifying potentially risky applicants.

These results establish a strong baseline and motivate the use of more advanced ensemble methods such as XGBoost and LightGBM.

## Logistic Regression Model Comparison

| Metric | Logistic Regression | Balanced Logistic Regression |
|----------|----------:|----------:|
| ROC-AUC | 0.7486 | 0.7482 |
| Accuracy | 0.92 | 0.69 |
| Precision (Default Class) | 0.58 | 0.16 |
| Recall (Default Class) | 0.01 | 0.68 |
| F1-Score (Default Class) | 0.02 | 0.26 |
| True Positives | 53 | 3,355 |
| False Negatives | 4,912 | 1,610 |
| False Positives | 39 | 17,568 |

### Key Findings

- The baseline Logistic Regression model achieved a reasonable ROC-AUC score but failed to identify most default cases.
- Applying class balancing dramatically improved recall for the default class, increasing it from 1% to 68%.
- The balanced model successfully identified 3,355 default cases compared to only 53 identified by the baseline model.
- This improvement came at the cost of lower precision and a larger number of false positives.
- ROC-AUC remained almost unchanged, indicating that class balancing primarily affected the decision threshold rather than the model's underlying ranking ability.
- Since credit risk prediction prioritizes identifying risky applicants, the balanced model provides a more useful baseline despite its lower precision.

## XGBoost Model Development

While Logistic Regression provided a useful baseline, it assumes a linear relationship between features and the target variable.

XGBoost is a gradient boosting algorithm capable of capturing complex nonlinear patterns and feature interactions, making it particularly effective for structured financial datasets.

The objective of this stage is to determine whether a more sophisticated ensemble model can improve discriminatory power beyond the Logistic Regression baseline.

In [ ]:
from xgboost import XGBClassifier

In [ ]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print(scale_pos_weight)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train,
    y_train
)

In [ ]:
xgb_preds = xgb_model.predict(
    X_test
)

xgb_probs = (
    xgb_model.predict_proba(
        X_test
    )[:, 1]
)

In [ ]:
xgb_auc = roc_auc_score(
    y_test,
    xgb_probs
)

print("XGBoost ROC-AUC:", xgb_auc)

In [ ]:
print(
    classification_report(
        y_test,
        xgb_preds
    )
)

In [ ]:
confusion_matrix(
    y_test,
    xgb_preds
)

In [ ]:
feature_importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_train.columns
)

feature_importance.sort_values(
    ascending=False
).head(20)

## XGBoost Evaluation Summary

XGBoost achieved the strongest performance observed so far, increasing ROC-AUC from 0.7486 (Logistic Regression) to 0.7595.

The model maintained strong recall for the default class while improving overall discriminatory power and F1-score.

Feature importance analysis revealed that several engineered variables, including EXT_SOURCE_MEAN, EMPLOYMENT_YEARS, EXT_SOURCE_1_MISSING, OWN_CAR_AGE_MISSING, and DAYS_EMPLOYED_PLACEHOLDER, contributed significantly to prediction performance.

These findings validate the feature engineering decisions made during preprocessing and demonstrate the effectiveness of gradient boosting methods for credit risk prediction.

## Model Performance Comparison

| Metric | Logistic Regression | Balanced Logistic Regression | XGBoost |
|----------|----------:|----------:|----------:|
| ROC-AUC | 0.7486 | 0.7482 | **0.7595** |
| Accuracy | **0.92** | 0.69 | 0.72 |
| Precision (Default Class) | **0.58** | 0.16 | 0.17 |
| Recall (Default Class) | 0.01 | **0.68** | 0.66 |
| F1-Score (Default Class) | 0.02 | 0.26 | **0.28** |

### Key Findings

- Logistic Regression achieved the highest accuracy, but largely failed to identify default cases due to severe class imbalance.
- Applying class balancing dramatically improved recall, increasing the model's ability to detect defaulters.
- XGBoost delivered the best overall performance, achieving the highest ROC-AUC and F1-score while maintaining strong recall.
- The improvement in ROC-AUC indicates that XGBoost is better at distinguishing between risky and non-risky applicants.
- Feature importance analysis further confirmed the value of engineered features created during preprocessing.